# Per-article BCQuality coverage (code-review)

Ad-hoc analysis of how the code-review gold dataset maps onto BCQuality
knowledge articles. Every finding is annotated with the article it derives
from (`ReviewComment.article`), and false-positive-guard entries carry their
association at entry level (`metadata.articles`). This notebook aggregates
those annotations via `bcbench.analysis.bcquality_article_coverage`.

Set `BCQUALITY_ROOT` (or edit the cell below) to point at a BCQuality checkout
to also surface articles with **zero** gold coverage; without it the report
covers declared articles only.

In [1]:
import os

from bcbench.analysis.bcquality_article_coverage import build_coverage_report, enumerate_inventory, resolve_bcquality_root
from bcbench.dataset import CodeReviewEntry
from bcbench.types import EvaluationCategory

entries = CodeReviewEntry.load(EvaluationCategory.CODE_REVIEW.dataset_path)

root = resolve_bcquality_root(os.environ.get("BCQUALITY_ROOT"))
inventory = enumerate_inventory(root) if root is not None else None

report = build_coverage_report(entries, inventory)
summary = f"{len(report.covered)} articles covered across {report.annotated_entries}/{report.total_entries} annotated entries"
if report.inventory_available:
    summary += f"; {len(report.zero_coverage)} of {report.inventory_size} articles have zero coverage"
print(summary)

9 articles covered across 17/130 annotated entries; 177 of 186 articles have zero coverage


## Coverage by domain

In [2]:
import pandas as pd

domains = sorted({c.domain for c in report.covered} | {a.split("/", 1)[0] for a in report.zero_coverage})
rows = []
for domain in domains:
    covered = [c for c in report.covered if c.domain == domain]
    zero = [a for a in report.zero_coverage if a.split("/", 1)[0] == domain]
    entry_ids = {entry_id for article in covered for entry_id in article.entry_ids}
    row = {"Domain": domain, "Covered": len(covered), "Gold entries": len(entry_ids)}
    if report.inventory_available:
        row["Inventory"] = len(covered) + len(zero)
        row["Zero-cov"] = len(zero)
    rows.append(row)

coverage_df = pd.DataFrame(rows)
print(coverage_df.to_string(index=False))

          Domain  Covered  Gold entries  Inventory  Zero-cov
breaking-changes        0             0          6         6
  error-handling        0             0          3         3
          events        0             0         16        16
      interfaces        0             0          3         3
     performance        0             0         42        42
         privacy        0             0         17        17
        security        9            17         18         9
           style        0             0         35        35
         testing        0             0          1         1
              ui        0             0         21        21
         upgrade        0             0         18        18
    web-services        0             0          6         6


## Covered articles and the entries that exercise them

In [3]:
article_df = (
    pd.DataFrame([{"Article": c.article, "Entries": c.count, "Instance IDs": ", ".join(c.entry_ids)} for c in report.covered]).sort_values(["Article"])
    if report.covered
    else pd.DataFrame(columns=["Article", "Entries", "Instance IDs"])
)
print(article_df.to_string(index=False))

                                                     Article  Entries                                                                                            Instance IDs
                      security/al-has-no-built-in-htmlencode        1                                                                                 synthetic__security-016
                 security/inherent-permissions-minimal-grant        1                                                                                 synthetic__security-006
  security/nondebuggable-required-when-unwrapping-secrettext        3                          synthetic__security-003, synthetic__security-012, synthetic__security-clean-03
               security/permission-set-avoid-wildcard-grants        1                                                                                 synthetic__security-005
security/recordref-open-with-caller-table-must-not-be-public        1                                                             

## Gaps: unknown slugs, zero-coverage articles, unannotated entries

In [4]:
if report.unknown_articles:
    print(f"Unknown article slugs not in inventory ({len(report.unknown_articles)}):")
    for a in report.unknown_articles:
        print(f"  - {a}")

if report.zero_coverage:
    print(f"\nZero-coverage articles ({len(report.zero_coverage)}):")
    for a in report.zero_coverage:
        print(f"  - {a}")

if report.unannotated_entry_ids:
    print(f"\nUnannotated entries ({len(report.unannotated_entry_ids)}):")
    for e in report.unannotated_entry_ids:
        print(f"  - {e}")


Zero-coverage articles (177):
  - breaking-changes/choose-access-modifiers-deliberately
  - breaking-changes/deprecate-public-members-with-the-obsolete-lifecycle
  - breaking-changes/do-not-change-published-procedure-signatures
  - breaking-changes/do-not-expose-sensitive-data-through-public-api
  - breaking-changes/do-not-modify-code-already-marked-obsolete
  - breaking-changes/obsolete-table-fields-instead-of-deleting-them
  - error-handling/collect-validation-errors-with-errorbehavior
  - error-handling/errortype-internal-vs-client-for-diagnostics
  - error-handling/prefer-errorinfo-for-actionable-errors
  - events/add-new-event-parameters-at-the-end
  - events/avoid-loosely-typed-event-parameters
  - events/avoid-raising-events-inside-try-functions
  - events/choose-static-vs-manual-subscribers-deliberately
  - events/do-not-add-ishandled-to-an-existing-event
  - events/do-not-bypass-critical-operations-with-ishandled
  - events/do-not-publish-events-inside-loops
  - events/initia